# Notebook 4 - Recency-Balanced Negatives
### Scientific Reports 35109da0 | removing the last confound

### What the normalisation test showed

| model | PR-AUC |
|---|---|
| **size proxies only** (6 features) | **0.882** |
| full model, size removed entirely | 0.850 |

The detector does not beat a size-only baseline, so the reported 0.800 F1 is largely a
package-size effect. Reviewer 3 asked for exactly this test, so it will be run against
the paper whether or not it appears in it.

**But version context itself is sound.** With size fully removed:

| | PR-AUC |
|---|---|
| correct predecessor | **0.850** |
| shuffled predecessor | **0.669** |

A gain of **+0.181** that owes nothing to size.

### The actual defect

Every negative is *older* than the malicious pair it controls for:

* malicious pair `(v[i-1], v[i])` - late in the package history
* matched negative `(v[i-2], v[i-1])` - earlier in the same history

Packages grow over time, so the malicious pair is larger in **74.8%** of packages
(median AST nodes 8,044 against 467). The model learns *newer*, not *malicious*.

### The fix

Add **post-incident** negatives `(v[i+1], v[i+2])`, drawn from *after* the compromise.
Those are newer than the malicious pair, so natural growth now works **against** the model.
With pre-incident and post-incident controls both present, recency is no longer tied to
the label and any surviving signal is real.

Every feature is also expressed as a size-independent rate before modelling.

**Runtime ~90 minutes.** Checkpointed to Drive; re-run to resume.

## Setup

In [ ]:
!pip install esprima -q
import os, io, re, ast, sys, math, json, time, glob, shutil, zipfile, tarfile, warnings
from collections import Counter
warnings.filterwarnings("ignore")
import requests, numpy as np, pandas as pd

SEED = 42
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"): drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/paper_revision"
except Exception:
    WORK = os.path.abspath("./paper_revision")
os.makedirs(WORK, exist_ok=True)
P = lambda n: os.path.join(WORK, n)

def log(m):
    line = f"[{time.strftime('%H:%M:%S')}] {m}"
    print(line, flush=True)
    try: open(P("pipeline_log.txt"), "a").write(line + "\n")
    except Exception: pass

KEY_STR = {"package":str,"ecosystem":str,"version_before":str,"version_after":str}
paired = pd.read_csv(P("features_v3_paired.csv"), dtype={"package":str,"ecosystem":str})
usable = pd.read_csv(P("merged_dataset_usable.csv"), dtype=KEY_STR)
paired["label"] = paired.label.astype(int)
log(f"features_v3_paired {paired.shape} | packages {paired.package.nunique()}")
log(str(paired.groupby(["ecosystem","label"]).size()))

## Registry access and feature extraction

Byte-identical to notebook 3 so every new row stays comparable with the 813 already built.

In [ ]:
NPM_URL  = "https://registry.npmjs.org/{p}"
PYPI_URL = "https://pypi.org/pypi/{p}/json"
HEADERS  = {"User-Agent": "supply-chain-research/1.0 (academic)"}
DELAY, RETRIES = 0.4, 3

def _request(url, want_json, timeout):
    for a in range(RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout); time.sleep(DELAY)
            if r.status_code == 200: return r.json() if want_json else r.content
            if r.status_code in (429,500,502,503): time.sleep(2**a); continue
            return None
        except requests.RequestException: time.sleep(2**a)
        except ValueError: return None
    return None

fetch_npm  = lambda p: _request(NPM_URL.format(p=p), True, 15)
fetch_pypi = lambda p: _request(PYPI_URL.format(p=p), True, 15)
dl         = lambda u: _request(u, False, 30) if u else None

def npm_versions(m):
    t = m.get("time", {}) or {}
    vs = [v for v in (m.get("versions", {}) or {}) if v not in ("created","modified") and v != "0.0.1-security"]
    pr = [(v, t.get(v)) for v in vs if t.get(v)]; pr.sort(key=lambda x: x[1]); return pr

def pypi_versions(m):
    pr = []
    for v, fs in (m.get("releases", {}) or {}).items():
        if not fs: continue
        ut = fs[0].get("upload_time_iso_8601") or fs[0].get("upload_time")
        if ut: pr.append((v, ut))
    pr.sort(key=lambda x: x[1]); return pr

def versions_for(pkg, eco):
    m = fetch_npm(pkg) if eco == "npm" else fetch_pypi(pkg)
    if not m: return None
    return (npm_versions(m) if eco == "npm" else pypi_versions(m))

def tarball_url(pkg, eco, v):
    if eco == "npm":
        m = fetch_npm(pkg)
        try: return m["versions"][v]["dist"]["tarball"]
        except (KeyError, TypeError): return None
    m = fetch_pypi(pkg)
    try:
        fs = m["releases"].get(v, [])
        if not fs: return None
        sd = [f for f in fs if f["url"].endswith((".tar.gz",".zip"))]
        return (sd[0] if sd else fs[0])["url"]
    except (KeyError, TypeError): return None

SUSPICIOUS = {
  "net_http":        re.compile(r"\b(https?://|requests\.(get|post)|urllib|fetch\(|XMLHttpRequest|http\.request|axios)", re.I),
  "subprocess_exec": re.compile(r"\b(subprocess|child_process|os\.system|exec\(|eval\(|spawn\(|popen)", re.I),
  "env_cred_access": re.compile(r"(process\.env|os\.environ|getenv|\.aws/credentials|id_rsa|\.npmrc|\.pypirc|SSH_|API_KEY|SECRET)", re.I),
  "filesystem":      re.compile(r"\b(fs\.(write|read|unlink)|open\([^)]*['\"]w|shutil|os\.remove|writeFile)", re.I),
  "encoded_blob":    re.compile(r"(base64|atob|btoa|fromCharCode|eval\(.*decode|exec\(.*decode)", re.I),
  "install_hook":    re.compile(r"(preinstall|postinstall|setup\.py|cmdclass|__import__)", re.I),
  "obfuscation":     re.compile(r"(_0x[0-9a-f]{4,}|[A-Za-z0-9+/]{120,})", re.I),
}
try:
    import esprima; HAVE_ES = True
except ImportError:
    HAVE_ES = False; log("esprima unavailable - JS AST uses a token proxy")

MAXP = 400_000
def entropy(s):
    if not s: return 0.0
    n = len(s); return -sum((c/n)*math.log2(c/n) for c in Counter(s).values())

def extract_files(arch, eco):
    if not arch: return []
    exts = (".js",".ts",".json",".mjs",".cjs") if eco=="npm" else (".py",".cfg",".toml",".txt")
    out = []
    try:
        try:
            with tarfile.open(fileobj=io.BytesIO(arch), mode="r:gz") as t:
                for m in t.getmembers():
                    if m.isfile() and m.name.endswith(exts):
                        f = t.extractfile(m)
                        if f: out.append((m.name, f.read().decode("utf-8","ignore")))
        except tarfile.ReadError:
            with zipfile.ZipFile(io.BytesIO(arch)) as z:
                for n in z.namelist():
                    if n.endswith(exts): out.append((n, z.read(n).decode("utf-8","ignore")))
    except Exception: return []
    return out

def _cnt(node):
    if isinstance(node, dict): return 1 + sum(_cnt(v) for k,v in node.items() if k!="type")
    if isinstance(node, list): return sum(_cnt(v) for v in node)
    return 0
def tok_proxy(t): return len(re.findall(r"[A-Za-z_$][\w$]*|\d+|\S", t[:MAXP]))

def ast_nodes(files, eco):
    total, ok, seen = 0, 0, 0
    for name, txt in files:
        if eco == "npm":
            if not name.endswith((".js",".mjs",".cjs")): continue
            seen += 1; done = False
            if HAVE_ES:
                for parse in (esprima.parseScript, esprima.parseModule):
                    for kw in ({"tolerant": True}, {}):
                        try:
                            total += _cnt(parse(txt[:MAXP], **kw).toDict()); ok += 1; done = True; break
                        except TypeError: continue
                        except Exception: break
                    if done: break
            if not done: total += tok_proxy(txt)
        else:
            if not name.endswith(".py"): continue
            seen += 1
            try: total += sum(1 for _ in ast.walk(ast.parse(txt[:MAXP]))); ok += 1
            except Exception: total += tok_proxy(txt)
    return total, ok, seen

def pcounts(c): return {k: len(rx.findall(c)) for k,rx in SUSPICIOUS.items()}
MIN_CHARS = 30

def features_for_pair(pkg, eco, vb, va):
    fa = extract_files(dl(tarball_url(pkg, eco, va)), eco)
    if sum(len(t) for _,t in fa) < MIN_CHARS: return None, "after_download_failed"
    fb = extract_files(dl(tarball_url(pkg, eco, vb)), eco)
    if sum(len(t) for _,t in fb) < MIN_CHARS: return None, "before_download_failed"
    cb = "\n".join(t for _,t in fb); ca = "\n".join(t for _,t in fa)
    bc, ac = pcounts(cb), pcounts(ca)
    f = {}
    for k in SUSPICIOUS:
        f[f"delta_{k}"] = ac[k]-bc[k]; f[f"after_{k}"] = ac[k]
    f["delta_loc"]  = ca.count("\n") - cb.count("\n")
    f["delta_char"] = len(ca) - len(cb)
    f["entropy_before"] = round(entropy(cb[:50000]),3)
    f["entropy_after"]  = round(entropy(ca[:50000]),3)
    f["delta_entropy"]  = round(f["entropy_after"]-f["entropy_before"],3)
    lt = lambda c: max((len(t) for t in re.findall(r"\S+", c)), default=0)
    f["delta_longest_token"] = lt(ca)-lt(cb)
    na, oka, sa = ast_nodes(fa, eco); nb, okb, sb = ast_nodes(fb, eco)
    f["ast_nodes_after"]=na; f["ast_nodes_before"]=nb; f["delta_ast_nodes"]=na-nb
    f["ast_parse_rate"]=round(oka/sa,3) if sa else 0.0
    kb = max(len(ca)/1024.0, 1.0)
    f["size_kb_after"] = round(len(ca)/1024.0, 2)
    for k in SUSPICIOUS: f[f"norm_after_{k}"] = round(f[f"after_{k}"]/kb, 4)
    return f, "ok"

META = ["package","ecosystem","label","version_before","version_after","version_after_date","origin"]
FEATS = ([f"delta_{k}" for k in SUSPICIOUS]+[f"after_{k}" for k in SUSPICIOUS]+
         ["delta_loc","delta_char","entropy_before","entropy_after","delta_entropy",
          "delta_longest_token","ast_nodes_after","ast_nodes_before","delta_ast_nodes",
          "ast_parse_rate","size_kb_after"]+[f"norm_after_{k}" for k in SUSPICIOUS])
ALL_COLS = FEATS + META

def flush(rows, path):
    if rows: pd.DataFrame(rows).reindex(columns=ALL_COLS).to_csv(
        path, mode="a", header=not os.path.exists(path), index=False)

def load_ck(path):
    if not os.path.exists(path): return pd.DataFrame(columns=ALL_COLS)
    try: return pd.read_csv(path, dtype=KEY_STR)
    except Exception:
        shutil.move(path, path+".corrupt"); return pd.DataFrame(columns=ALL_COLS)

log("extraction ready - identical to notebook 3")

## Stage 1 - post-incident negatives

For each malicious version `v[i]`, the control is `(v[i+1], v[i+2])`: a normal update taken
from **after** the compromise, so it is *newer and larger* than the pair it controls for.

`(v[i], v[i+1])` is deliberately avoided as the primary choice - that is the remediation
release, whose delta is the *removal* of malicious code rather than an ordinary update.
It is collected only as a fallback and tagged separately so it can be excluded.

Many compromised packages are unpublished after an incident, so yield will be moderate.

In [ ]:
CK_POST = P("post_incident_negatives.csv")

mal_pairs = (usable[usable.download_status == "both_available"]
             [["package","ecosystem","version_before","version_after"]]
             .dropna()
             .drop_duplicates(subset=["package","ecosystem","version_after"])
             .reset_index(drop=True))
log(f"malicious pairs needing a post-incident control: {len(mal_pairs)}")

prev = load_ck(CK_POST)
done = {(str(a),str(b)) for a,b in zip(prev.package, prev.version_before)} if len(prev) else set()
if done: log(f"resuming - {len(done)} done")

rows, st = [], Counter()
try:
    for n, r in enumerate(mal_pairs.itertuples(index=False), 1):
        try:
            hist = versions_for(r.package, r.ecosystem)
            if not hist: st["package_unpublished"] += 1; continue
            vers = [v for v,_ in hist]
            vm = str(r.version_after)
            if vm not in vers: st["malicious_version_removed"] += 1; continue
            i = vers.index(vm)

            if i + 2 < len(vers):
                vb, va, tag = vers[i+1], vers[i+2], "post_incident_negative"
            elif i + 1 < len(vers):
                vb, va, tag = vers[i], vers[i+1], "post_remediation_negative"
            else:
                st["no_release_after_incident"] += 1; continue

            if (str(r.package), str(vb)) in done: st["skipped"] += 1; continue
            f, s = features_for_pair(r.package, r.ecosystem, vb, va)
            if f is None: st[s] += 1; continue
            f.update(package=r.package, ecosystem=r.ecosystem, label=0,
                     version_before=vb, version_after=va,
                     version_after_date=dict(hist)[va], origin=tag)
            rows.append(f); st[tag] += 1
        except Exception as e:
            st[f"err:{type(e).__name__}"] += 1
        if n % 25 == 0:
            flush(rows, CK_POST); rows = []
            log(f"  {n}/{len(mal_pairs)} {dict(st)}")
except KeyboardInterrupt: log("interrupted - progress saved")
finally: flush(rows, CK_POST)

post = load_ck(CK_POST)
if len(post):
    post = post.drop_duplicates(subset=["package","ecosystem","version_after"])
    post = post[post.label.astype(str) == "0"]
log(f"post-incident negatives built: {len(post)}")
if len(post): log(str(post.origin.value_counts()))
log(f"breakdown: {dict(st)}")

## Stage 2 - the recency-balanced dataset

Positives keep their pre-incident controls **and** gain post-incident ones. Recency is now
spread across both classes, so it can no longer stand in for the label.

In [ ]:
bal = pd.concat([paired, post], ignore_index=True, sort=False)
bal["label"] = pd.to_numeric(bal.label, errors="coerce").fillna(0).astype(int)
bal = bal.drop_duplicates(subset=["package","ecosystem","version_after"]).reset_index(drop=True)

# keep only packages that still contribute both classes
ok = bal.groupby(["package","ecosystem"]).label.nunique() > 1
keep = set(ok[ok].index)
bal = bal[[(p,e) in keep for p,e in zip(bal.package, bal.ecosystem)]].reset_index(drop=True)

log(f"balanced dataset: n={len(bal)}  packages={bal.package.nunique()}  "
    f"prevalence={bal.label.mean():.3f}")
log(str(bal.groupby(["ecosystem","label"]).size()))
log(str(bal.origin.value_counts()))
bal.to_csv(P("features_v4_balanced.csv"), index=False)
log("saved features_v4_balanced.csv")

### Confound diagnostic

Before modelling: is the malicious pair still systematically the larger one?
Under the old design it was, in 74.8% of packages. Anything near 50% means recency and size
no longer carry the label.

In [ ]:
rows = []
for (pkg, eco), sub in bal.groupby(["package","ecosystem"]):
    if sub.label.nunique() < 2: continue
    rows.append({"ast_pos": sub[sub.label==1].ast_nodes_after.mean(),
                 "ast_neg": sub[sub.label==0].ast_nodes_after.mean(),
                 "kb_pos":  sub[sub.label==1].size_kb_after.mean(),
                 "kb_neg":  sub[sub.label==0].size_kb_after.mean()})
dd = pd.DataFrame(rows)
log(f"packages compared: {len(dd)}")
log(f"malicious pair larger in AST nodes : {100*(dd.ast_pos>dd.ast_neg).mean():.1f}%   (was 74.8%)")
log(f"malicious pair larger in KB        : {100*(dd.kb_pos>dd.kb_neg).mean():.1f}%   (was 70.4%)")
log(f"median AST  pos={dd.ast_pos.median():.0f}  neg={dd.ast_neg.median():.0f}  (was 8044 vs 467)")

## Stage 3 - size-independent features

Raw counts and raw deltas both scale with package size. Every signal becomes a **density**
(occurrences per KB) and every change becomes a **relative** change, so a package that merely
grew between releases no longer resembles a compromise.

In [ ]:
SIG = list(SUSPICIOUS.keys())

def build_rates(d):
    kb_after  = d["size_kb_after"].clip(lower=1.0)
    kb_before = (d["size_kb_after"] - d["delta_char"]/1024.0).clip(lower=1.0)
    r = pd.DataFrame(index=d.index)
    for k in SIG:
        a_after  = d[f"after_{k}"]
        a_before = a_after - d[f"delta_{k}"]
        r[f"rate_after_{k}"]  = a_after / kb_after
        r[f"rate_before_{k}"] = a_before / kb_before
        r[f"rate_delta_{k}"]  = r[f"rate_after_{k}"] - r[f"rate_before_{k}"]
    r["rel_char"] = d["delta_char"] / (kb_before*1024.0)
    r["rel_loc"]  = d["delta_loc"]  / (kb_before*1024.0/40.0)
    r["rel_ast"]  = d["delta_ast_nodes"] / d["ast_nodes_before"].abs().clip(lower=1)
    r["entropy_after"]  = d["entropy_after"]
    r["entropy_before"] = d["entropy_before"]
    r["delta_entropy"]  = d["delta_entropy"]
    r["delta_longest_token"] = d["delta_longest_token"]
    return r.replace([np.inf,-np.inf], np.nan).fillna(0.0)

rates = build_rates(bal)
RATE_CAND = [c for c in rates.columns if c.startswith("rate_after_")] + ["entropy_after"]
RATE_PRED = ([c for c in rates.columns if c.startswith("rate_delta_")] +
             [c for c in rates.columns if c.startswith("rate_before_")] +
             ["entropy_before","delta_entropy","rel_char","rel_loc","rel_ast","delta_longest_token"])
RAW  = ([c for c in bal.columns if c.startswith("after_")] +
        [c for c in bal.columns if c.startswith("delta_")] + ["entropy_after","entropy_before"])
SIZE = ["size_kb_after","ast_nodes_after","ast_nodes_before",
        "delta_char","delta_loc","delta_ast_nodes"]
log(f"rate candidate {len(RATE_CAND)} | rate predecessor {len(RATE_PRED)} | size proxies {len(SIZE)}")

## Stage 4 - evaluation

Package-grouped CV, class weighting, three random seeds. The **size-only baseline is reported
first** - if the model cannot beat it, nothing else in the table matters.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score, confusion_matrix,
                             average_precision_score, roc_auc_score)

y, g = bal.label.values, bal.package.values
SEEDS = (42, 7, 123)

def run(X, name, store=None, results=None):
    f1s, prs, rocs, fprs, oof_last = [], [], [], [], None
    for s in SEEDS:
        oof = np.zeros(len(X))
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(X, y, g):
            pos = y[tr].sum()
            w = np.where(y[tr]==1, (len(tr)-pos)/max(pos,1), 1.0)
            m = HistGradientBoostingClassifier(max_iter=300, random_state=s)
            m.fit(X[tr], y[tr], sample_weight=w)
            oof[te] = m.predict_proba(X[te])[:,1]
        pred = (oof>=0.5).astype(int)
        tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
        f1s.append(f1_score(y,pred,zero_division=0)); prs.append(average_precision_score(y,oof))
        rocs.append(roc_auc_score(y,oof)); fprs.append(fp/(fp+tn) if fp+tn else 0.0)
        oof_last = oof
    r = {"f1":round(float(np.mean(f1s)),4), "f1_sd":round(float(np.std(f1s)),4),
         "pr_auc":round(float(np.mean(prs)),4), "roc_auc":round(float(np.mean(rocs)),4),
         "fpr":round(float(np.mean(fprs)),4)}
    log(f"  {name:34s} F1={r['f1']:.3f}+/-{r['f1_sd']:.3f}  PR-AUC={r['pr_auc']:.3f}  "
        f"ROC-AUC={r['roc_auc']:.3f}  FPR={r['fpr']:.3f}")
    if results is not None: results.setdefault(store,{})[name] = r
    return r, oof_last

R4 = {}
log(f"prevalence = {y.mean():.3f}  (chance PR-AUC)")
log("--- baselines ---")
eco = (bal.ecosystem=="npm").astype(int).values
tn,fp,fn,tp = confusion_matrix(y, eco, labels=[0,1]).ravel()
log(f"  {'ecosystem-only':34s} F1={f1_score(y,eco):.3f}  FPR={fp/(fp+tn):.3f}")
R4["ecosystem_only"] = {"f1": round(float(f1_score(y,eco)),4), "fpr": round(float(fp/(fp+tn)),4)}

r_size, _ = run(bal[SIZE].values, "SIZE PROXIES ONLY", "baselines", R4)
r_raw,  _ = run(bal[RAW].values,  "raw features", "baselines", R4)

log("--- size-independent ---")
r_cand, _   = run(rates[RATE_CAND].values, "rate candidate-only", "model", R4)
r_full, oof = run(rates[RATE_CAND+RATE_PRED].values, "rate full", "model", R4)

perm = np.random.RandomState(SEED).permutation(len(rates))
sh = rates.copy()
for c in RATE_PRED: sh[c] = sh[c].values[perm]
r_shuf, _ = run(sh[RATE_CAND+RATE_PRED].values, "rate full, shuffled predecessor", "model", R4)

vc_f1 = r_full["f1"] - r_shuf["f1"]
vc_pr = r_full["pr_auc"] - r_shuf["pr_auc"]
over  = r_full["pr_auc"] - r_size["pr_auc"]
log("="*78)
log(f"version context (size removed) : {vc_f1:+.4f} F1   {vc_pr:+.4f} PR-AUC")
log(f"model vs size-only baseline    : {over:+.4f} PR-AUC")
log("="*78)
R4["verdict"] = {"vc_gain_f1":round(vc_f1,4), "vc_gain_pr_auc":round(vc_pr,4),
                 "over_size_only_pr_auc":round(over,4)}
if over > 0.02 and vc_pr > 0.02:
    log("VERDICT: real signal beyond package size, and version context contributes.")
elif over <= 0.02:
    log("VERDICT: still not beating a size-only baseline.")
else:
    log("VERDICT: signal beyond size, but version context adds little once size is removed.")

### Per-ecosystem and operating points

In [ ]:
for e in ["npm","pypi"]:
    msk = (bal.ecosystem==e).values
    if msk.sum() and len(set(y[msk]))>1:
        pr = (oof[msk]>=0.5).astype(int)
        tn,fp,fn,tp = confusion_matrix(y[msk],pr,labels=[0,1]).ravel()
        log(f"  {e:5s} n={msk.sum():4d} F1={f1_score(y[msk],pr,zero_division=0):.3f} "
            f"Rec={tp/max(tp+fn,1):.3f} FPR={fp/max(fp+tn,1):.3f} "
            f"PR-AUC={average_precision_score(y[msk],oof[msk]):.3f}")
        R4.setdefault("per_ecosystem",{})[e] = {
            "n":int(msk.sum()), "f1":round(float(f1_score(y[msk],pr,zero_division=0)),4),
            "recall":round(float(tp/max(tp+fn,1)),4), "fpr":round(float(fp/max(fp+tn,1)),4),
            "pr_auc":round(float(average_precision_score(y[msk],oof[msk])),4)}

ops = []
for tgt in [0.01,0.05,0.10,0.20]:
    for th in np.unique(oof):
        tn,fp,fn,tp = confusion_matrix(y,(oof>=th).astype(int),labels=[0,1]).ravel()
        if fp+tn and fp/(fp+tn) <= tgt:
            ops.append({"target_fpr":tgt,"achieved_fpr":round(float(fp/(fp+tn)),4),
                        "recall":round(float(tp/max(tp+fn,1)),4),
                        "precision":round(float(tp/max(tp+fp,1)),4)}); break
log("operating points:"); [log(f"    {o}") for o in ops]
R4["operating_points"] = ops
R4["dataset"] = {"n":int(len(bal)), "packages":int(bal.package.nunique()),
                 "prevalence":round(float(y.mean()),4),
                 "by_eco_label":{f"{a}_{b}":int(c) for (a,b),c in
                                 bal.groupby(["ecosystem","label"]).size().items()},
                 "by_origin":{k:int(v) for k,v in bal.origin.value_counts().items()}}
json.dump(R4, open(P("revision_results_v4.json"),"w"), indent=2, default=str)
log("saved revision_results_v4.json")

---
## Reading the result

The single line that decides everything:

```
model vs size-only baseline : +X.XXXX PR-AUC
```

* **clearly positive** - the detector finds something package size does not, the confound is
  handled, and the numbers are defensible under the exact test Reviewer 3 specified.
* **near zero or negative** - the signal is size, and no amount of rewriting will survive
  review. Better to know now.

Send back `revision_results_v4.json` and `features_v4_balanced.csv`.